# III. Analyse exploratoire (EDA)

**Objectif**: Identifier les facteurs liés au churn et formuler des hypothèses business actionnables.

**Méthode**: Pour chaque variable, on compare la distribution entre churners et non-churners.
On travaille syr df_eda qui conserve les labels originaux (Yes/No) pour lisibilité.

**Fonctions utilisées**: src/features.py

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.features import (
    add_tenure_group,
    add_charge_ratio,
    add_is_new_customer,
    add_nb_services
    )

pd.set_option('display.max.rows', 10)
DATA_PATH = '../data/processed/telco_eda.csv'


In [3]:
df_eda = pd.read_csv(DATA_PATH)
print(f'Dataset EDA chargé: {df_eda.shape}\n')
print(f'Taux de churn: {df_eda['Churn_num'].mean()*100:.1f}%')

Dataset EDA chargé: (7043, 22)

Taux de churn: 26.5%


## 1. Churn par type de contrat

In [4]:
contract_churn = df_eda.groupby('Contract')['Churn_num'].agg(['mean', 'count']).round(3)
contract_churn.columns = ['Taux de churn', 'Nb clients']
contract_churn['Taux (%)'] = (contract_churn['Taux de churn']*100).round(1)
contract_churn = contract_churn.sort_values('Taux de churn', ascending=False)

print('Churn par type de contrat')
print(contract_churn)


Churn par type de contrat
                Taux de churn  Nb clients  Taux (%)
Contract                                           
Month-to-month          0.427        3875      42.7
One year                0.113        1473      11.3
Two year                0.028        1695       2.8


In [5]:
ratio = contract_churn['Taux (%)'].iloc[0] / contract_churn['Taux (%)'].iloc[-1]
print(f'Insight: contrat mensuel = {ratio:.0f}x plus de churn que contrat 2 ans')

Insight: contrat mensuel = 15x plus de churn que contrat 2 ans


## 2. Churn vs ancienneté

In [6]:
# add_tenure_group vient de src/features.py
df_eda = add_tenure_group(df_eda)

tenure_churn = df_eda.groupby('tenure_group', observed=True)['Churn_num'].agg(['mean', 'count'])
tenure_churn.columns = ['Taux de churn', 'Nb clients']
tenure_churn['Taux (%)'] = (tenure_churn['Taux de churn']*100).round(1)

print('Churn par ancienneté')
print(tenure_churn)

Churn par ancienneté
              Taux de churn  Nb clients  Taux (%)
tenure_group                                     
0-1 an             0.474382        2186      47.4
1-2 ans            0.287109        1024      28.7
2-4 ans            0.203890        1594      20.4
4-6 ans            0.095132        2239       9.5


## 3. Churn vs charges mensuelles

In [7]:
charges_stats = df_eda.groupby('Churn')['MonthlyCharges'].describe().round(2)
print(f'MonthlyCharges par group Churn:\n {charges_stats}')

MonthlyCharges par group Churn:
         count   mean    std    min    25%    50%   75%     max
Churn                                                         
No     5174.0  61.27  31.09  18.25  25.10  64.43  88.4  118.75
Yes    1869.0  74.44  24.67  18.85  56.15  79.65  94.2  118.35


In [9]:
diff = charges_stats.loc['Yes', 'mean'] - charges_stats.loc['No', 'mean']
print(f'Les churners paient en moyenne {diff:.1f} €/mois de plus')

Les churners paient en moyenne 13.2 €/mois de plus


## 4. Churn par service internet

In [11]:
internet_churn = df_eda.groupby('InternetService')['Churn_num'].mean().sort_values(ascending=False)
print("Churn par type d'internet")

for service, rate in internet_churn.items():
    print(f"{service:20s} : {rate:.1f}%")

Churn par type d'internet
Fiber optic          : 0.4%
DSL                  : 0.2%
No                   : 0.1%
